# Week 2 — Task 3: Cognitive Load and the Data-Ink Ratio

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Datasets:** `car_crashes.csv` (51 rows, 50 states + DC) and `tips.csv`.

**All charts built with `fig, ax = plt.subplots()`. OO API throughout.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

In [ ]:
crashes = pd.read_csv('data/car_crashes.csv')
tips = pd.read_csv('data/tips.csv')
print(f'car_crashes shape: {crashes.shape}')
print(f'Columns: {crashes.columns.tolist()}')
crashes.head()

## Step 1 — Overloaded Chart (BAD EXAMPLE)

Purposely pile on every form of extraneous load: 51 colours, rotated labels, heavy gridlines, box frame, 51-entry legend, and an axis-name title.

In [ ]:
n = len(crashes)
cmap = plt.cm.get_cmap('tab20', n)

fig, ax = plt.subplots(figsize=(20, 10))
for i, (_, row) in enumerate(crashes.iterrows()):
    ax.bar(row['abbrev'], row['total'], color=cmap(i), alpha=0.9)

ax.set_xlabel('State Abbreviation', fontsize=12)
ax.set_ylabel('Total Fatal Collisions per 1B Miles', fontsize=12)
ax.set_title('BAD EXAMPLE: Total fatal collisions per state — abbrev vs total', fontsize=12)
ax.set_xticklabels(crashes['abbrev'], rotation=75, fontsize=7)
ax.grid(True, axis='both', color='black', lw=1.2)
for sp in ('top', 'bottom', 'left', 'right'):
    ax.spines[sp].set_visible(True)
    ax.spines[sp].set_linewidth(2)

legend_patches = [mpatches.Patch(color=cmap(i), label=abbrev)
                  for i, abbrev in enumerate(crashes['abbrev'])]
ax.legend(handles=legend_patches, ncol=4, fontsize=7, title='State',
          loc='upper right', frameon=True)

plt.tight_layout()
plt.savefig('task3_bad_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Load Classification and Data-Ink Ratio

| Element | Type | Reason |
|---|---|---|
| Bar height | **Germane** | Directly encodes the data value — it is the data. |
| 51 different bar colours | **Extraneous** | Colour adds nothing; state identity is already on x-axis. |
| 51-entry legend | **Extraneous** | Redundant with x-axis labels; requires serial search to use. |
| Heavy grid on both axes | **Extraneous** | Adds ink without helping comparison; a lighter x-grid only would be enough. |
| Box frame (all 4 spines) | **Extraneous** | Convention only; top/right spines carry no data. |
| Rotated x-axis labels | **Intrinsic** | The state names must appear somewhere; rotation is a necessary cost. |
| Axis title "abbrev vs total" | **Extraneous** | Restates the axis labels; adds zero information. |
| Drop shadow / 3-D | Not added | Backend limitation. |

**Data-ink ratio estimate:**
- Ink used to show data: ~51 bars × their height = ~51 rectangular patches. Each bar ≈ 4 ink units (4 sides plus fill proportional to height). Total data ink ≈ 204 units.
- Total ink: data ink + 51-colour legend (~51×6=306 units) + heavy gridlines (~200 units) + 4-sided frame (~40 units) + rotated labels (~51×4=204 units) ≈ 954 units.
- **Estimated ratio: 204 / 954 ≈ 0.21** — roughly 4 out of every 5 ink units carry no data.

## Step 3 — Five-Panel Progressive Declutter

In [ ]:
crashes_sorted = crashes.sort_values('total', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 5, figsize=(25, 7))

# Panel 1: Remove legend
ax = axes[0]
for i, (_, row) in enumerate(crashes.iterrows()):
    ax.bar(row['abbrev'], row['total'], color=cmap(i), alpha=0.9)
ax.set_title('1: Legend removed\n(extraneous — redundant with x-axis)', fontsize=7.5)
ax.set_xticklabels(crashes['abbrev'], rotation=75, fontsize=5)
ax.grid(True, axis='both', color='black', lw=1.2)
for sp in ('top', 'bottom', 'left', 'right'):
    ax.spines[sp].set_visible(True)

# Panel 2: Remove frame and y-gridlines
ax = axes[1]
for i, (_, row) in enumerate(crashes.iterrows()):
    ax.bar(row['abbrev'], row['total'], color=cmap(i), alpha=0.9)
ax.set_title('2: Frame + y-grid removed\n(extraneous)', fontsize=7.5)
ax.set_xticklabels(crashes['abbrev'], rotation=75, fontsize=5)
ax.yaxis.grid(True, color='#cccccc', lw=0.7)
ax.xaxis.grid(False)
for sp in ('top', 'right', 'bottom'):
    ax.spines[sp].set_visible(False)

# Panel 3: Single hue
ax = axes[2]
for _, row in crashes.iterrows():
    ax.bar(row['abbrev'], row['total'], color='#4575b4', alpha=0.85)
ax.set_title('3: Single hue\n(51 colours were extraneous)', fontsize=7.5)
ax.set_xticklabels(crashes['abbrev'], rotation=75, fontsize=5)
ax.yaxis.grid(True, color='#cccccc', lw=0.7)
for sp in ('top', 'right', 'bottom'):
    ax.spines[sp].set_visible(False)

# Panel 4: Sorted bars
ax = axes[3]
for _, row in crashes_sorted.iterrows():
    ax.bar(row['abbrev'], row['total'], color='#4575b4', alpha=0.85)
ax.set_title('4: Sorted by total\n(zero ink added; highest value now findable)', fontsize=7.5)
ax.set_xticklabels(crashes_sorted['abbrev'], rotation=75, fontsize=5)
ax.yaxis.grid(True, color='#cccccc', lw=0.7)
for sp in ('top', 'right', 'bottom'):
    ax.spines[sp].set_visible(False)

# Panel 5: Label only what matters — top 5 highlighted
top5 = crashes_sorted.head(5)['abbrev'].tolist()
ax = axes[4]
for _, row in crashes_sorted.iterrows():
    highlight = row['abbrev'] in top5
    color = '#d73027' if highlight else '#aaaaaa'
    ax.bar(row['abbrev'], row['total'], color=color, alpha=0.85)
ax.set_title('5: Highlight only top 5\n(grey = context; one strong colour = message)', fontsize=7.5)
ax.set_xticklabels(crashes_sorted['abbrev'], rotation=75, fontsize=5)
ax.yaxis.grid(True, color='#eeeeee', lw=0.5)
for sp in ('top', 'right', 'bottom'):
    ax.spines[sp].set_visible(False)

fig.suptitle('Progressive declutter: same data, five steps — each step removes extraneous load',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('task3_declutter_steps.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Respect the 4±1 Limit: Chunked Version

Working memory holds ~4 chunks. 51 states exceed this; we must chunk.

**Chosen approach:** Group into 5 US census regions (Northeast, South, Midwest, West, and DC-separately). Show the regional mean and let the reader focus on 5 things instead of 51.

**What this approach hides:** Individual state variation within a region. A state like Wyoming that is far above its regional mean becomes invisible once aggregated. Chunking always hides something — that is the honest cost.

In [ ]:
region_map = {
    'ME':'Northeast','NH':'Northeast','VT':'Northeast','MA':'Northeast','RI':'Northeast',
    'CT':'Northeast','NY':'Northeast','NJ':'Northeast','PA':'Northeast',
    'OH':'Midwest','MI':'Midwest','IN':'Midwest','IL':'Midwest','WI':'Midwest',
    'MN':'Midwest','IA':'Midwest','MO':'Midwest','ND':'Midwest','SD':'Midwest',
    'NE':'Midwest','KS':'Midwest',
    'DE':'South','MD':'South','DC':'South','VA':'South','WV':'South','NC':'South',
    'SC':'South','GA':'South','FL':'South','KY':'South','TN':'South','AL':'South',
    'MS':'South','AR':'South','LA':'South','OK':'South','TX':'South',
    'MT':'West','ID':'West','WY':'West','CO':'West','NM':'West','AZ':'West',
    'UT':'West','NV':'West','WA':'West','OR':'West','CA':'West','AK':'West','HI':'West'
}
crashes['region'] = crashes['abbrev'].map(region_map).fillna('Other')
region_means = crashes.groupby('region')['total'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors_region = ['#d73027' if r == region_means.index[0] else '#4575b4'
                 for r in region_means.index]
ax.bar(region_means.index, region_means.values, color=colors_region, alpha=0.85, edgecolor='white')
ax.set_title(
    'West region has highest average fatal collision rate\n'
    '(n=51; grouped into 5 census regions; within-region variation hidden)',
    loc='left'
)
ax.set_ylabel('Average total fatal collisions per 1B miles')
ax.set_ylim(0, 25)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task3_chunked.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Over-Stripped Chart (BAD EXAMPLE)

Remove too much — no axis labels, no tick values, no units — and show why maximising the data-ink ratio is not always right.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for _, row in crashes_sorted.iterrows():
    ax.bar(row['abbrev'], row['total'], color='#4575b4', alpha=0.85)

# Remove everything
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('BAD EXAMPLE: Over-stripped — no tick values, no labels, no units. What is the y-axis? Unknown.')
for sp in ('top', 'right', 'left', 'bottom'):
    ax.spines[sp].set_visible(False)

plt.tight_layout()
plt.savefig('task3_over_stripped.png', dpi=150, bbox_inches='tight')
plt.show()

print('Stopping rule to give a colleague:')
print('Remove ink until removing one more piece would make the chart harder to read or force the reader to guess what a value is — then stop.')

## Bonus — Adding Ink Can Lower Cognitive Load (`tips.csv`)

**Preattentive attribute:** Hue (lunch vs dinner).  
**Gestalt principle:** Proximity (within-day bars are grouped).  
**Ink addition that helps:** A direct value label on each bar. This saves the reader the mental calculation of reading the y-value from the axis — one less chunk in working memory.

**Why 'maximise data-ink' is a heuristic, not a rule:** The data-ink formula counts all ink, but not all ink does equal cognitive work. A direct label *replaces* a mental step (axis-to-bar comparison), so adding it lowers extraneous load even though it raises total ink. Tufte's ratio is a design heuristic for reducing decoration — it should not be applied mechanically to every pixel.

In [ ]:
mean_tip = tips.groupby(['day', 'time'])['tip'].mean().reset_index()
day_order = ['Thur', 'Fri', 'Sat', 'Sun']
mean_tip['day'] = pd.Categorical(mean_tip['day'], categories=day_order, ordered=True)
mean_tip = mean_tip.sort_values('day')

x = np.arange(len(day_order))
width = 0.3
lunch  = mean_tip[mean_tip['time']=='Lunch'].set_index('day').reindex(day_order)
dinner = mean_tip[mean_tip['time']=='Dinner'].set_index('day').reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Without labels — reader must use axis
axes[0].bar(x - width/2, lunch['tip'].fillna(0),   width, color='#4575b4', alpha=0.85, label='Lunch')
axes[0].bar(x + width/2, dinner['tip'].fillna(0), width, color='#d73027', alpha=0.85, label='Dinner')
axes[0].set_xticks(x)
axes[0].set_xticklabels(day_order)
axes[0].set_title('Without direct labels — reader must decode axis ticks', loc='left')
axes[0].set_ylabel('Average tip ($)')
axes[0].legend(frameon=False)
for sp in ('top','right'):
    axes[0].spines[sp].set_visible(False)

# With labels — reader gets value directly
bars_l = axes[1].bar(x - width/2, lunch['tip'].fillna(0),   width, color='#4575b4', alpha=0.85, label='Lunch')
bars_d = axes[1].bar(x + width/2, dinner['tip'].fillna(0), width, color='#d73027', alpha=0.85, label='Dinner')
axes[1].set_xticks(x)
axes[1].set_xticklabels(day_order)
axes[1].set_title('With direct labels — value is readable at a glance (lower cognitive load)', loc='left')
axes[1].legend(frameon=False)
for bar, val in [(b, lunch['tip'].fillna(0).values[i])
                 for i, b in enumerate(bars_l)]:
    if val > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, val+0.04, f'${val:.2f}',
                     ha='center', fontsize=7.5)
for bar, val in [(b, dinner['tip'].fillna(0).values[i])
                 for i, b in enumerate(bars_d)]:
    if val > 0:
        axes[1].text(bar.get_x()+bar.get_width()/2, val+0.04, f'${val:.2f}',
                     ha='center', fontsize=7.5)
for sp in ('top','right'):
    axes[1].spines[sp].set_visible(False)

fig.suptitle('Adding ink (direct labels) lowers cognitive load by removing the axis-decoding step',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('task3_tips_labels.png', dpi=150, bbox_inches='tight')
plt.show()